<a href="https://colab.research.google.com/github/AlwayszZZZ/4131Downloads0302/blob/main/0428Lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Install Dependencies
安装所有必要的依赖库。
注意：`vllm` 是推理服务组件，**微调阶段不需要**，此处不安装以避免版本冲突。

In [ ]:
# 安装微调所需的全部依赖包（-q 减少安装日志）
# datasets      : 数据集加载
# transformers   : Hugging Face 模型框架
# peft           : 参数高效微调（LoRA 等）
# trl            : SFT / RLHF 训练工具
# accelerate     : 多 GPU / 混合精度加速
# bitsandbytes   : 4-bit / 8-bit 量化，大幅节省显存
# sentencepiece  : Qwen tokenizer 的底层依赖
# huggingface_hub: 登录与模型下载
!pip install -q -U \
    datasets \
    transformers \
    peft \
    trl \
    accelerate \
    bitsandbytes \
    sentencepiece \
    huggingface_hub


## Step 2: Hugging Face Login
登录 Hugging Face，用于下载 Qwen 模型权重。
请前往 https://huggingface.co/settings/tokens 创建一个 **Read** 类型的 Access Token，粘贴到下方弹窗中。

In [ ]:
# 调用 login() 后会弹出 token 输入框
# 若已在 Colab Secrets 中设置了 HF_TOKEN，也可以改用：
#   from google.colab import userdata; login(token=userdata.get('HF_TOKEN'))
from huggingface_hub import login
login()


## Step 3: Mount Google Drive (Optional but Recommended)
挂载 Google Drive，用于**持久化**保存数据集和训练产物。
Colab 运行时断开后 `/content/` 下的文件会丢失；建议把重要文件保存到 Drive。
如果不需要持久化，可跳过此 cell，所有文件将存放在 `/content/`（断开即丢失）。

In [ ]:
# 挂载 Google Drive 到 /content/drive
# 挂载成功后，可在 /content/drive/MyDrive/ 下读写文件
from google.colab import drive
drive.mount('/content/drive')


## Step 4: Prepare Dataset
从 Hugging Face 下载食谱数据集，转换为 SFTTrainer 所需的 **messages 对话格式**，保存为 JSONL 文件。
如果你已有 `food_recipes.jsonl`（每行含 `messages` 字段），可将其上传到下方 `DATA_PATH` 并跳过此 cell。

In [ ]:
import os, json
from datasets import load_dataset

# 数据保存路径（可改为 Drive 路径，如 /content/drive/MyDrive/data/food_recipes.jsonl）
DATA_DIR  = "/content/data"
DATA_PATH = f"{DATA_DIR}/food_recipes.jsonl"
os.makedirs(DATA_DIR, exist_ok=True)

# 从 Hugging Face 下载食谱数据集（首次执行会下载几百 MB）
print("正在下载数据集...")
ds = load_dataset("Hieu-Pham/kaggle_food_recipes", split="train")
print(f"数据集总大小：{len(ds)} 条，字段：{ds.column_names}")

# 将原始字段转换为 messages 格式（system + user + assistant 三段对话）
# Qwen2.5 使用 ChatML 格式，SFTTrainer 要求每条数据包含 messages 字段
def to_messages(example):
    title        = example.get("Title", "") or example.get("title", "")
    instructions = (example.get("Instructions", "")
                    or example.get("instructions", "")
                    or example.get("Directions", "") or "")
    ingredients  = (example.get("Ingredients", "")
                    or example.get("ingredients", "") or "")
    user_content = f"Please give me a recipe for: {title}"
    if ingredients:
        user_content += f"\nIngredients available: {ingredients}"
    return {"messages": [
        {"role": "system",    "content": "You are a professional cooking assistant. Provide clear and detailed recipe instructions."},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": instructions or "No instructions available."},
    ]}

# 转换并过滤掉 instructions 过短的记录
ds_converted = ds.map(to_messages, remove_columns=ds.column_names)
ds_converted = ds_converted.filter(lambda x: len(x["messages"][2]["content"]) > 20)
print(f"过滤后数据量：{len(ds_converted)} 条")

# 保存为 JSONL 文件
with open(DATA_PATH, "w", encoding="utf-8") as f:
    for item in ds_converted:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print(f"✅ 数据集已保存到 {DATA_PATH}")


## Step 5: Check Dataset
读取 JSONL 文件，显示前 2 条记录，确认 `messages` 格式正确后再开始训练。

In [ ]:
import json

DATA_PATH = "/content/data/food_recipes.jsonl"  # 与上一步保持一致

print(f"=== 前 2 条数据记录预览（来自 {DATA_PATH} ）===")
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 2:
            break
        record = json.loads(line)
        print(f"\n[Record {i + 1}]")
        for msg in record["messages"]:
            # 内容太长时截断显示，避免输出刷屏
            content = msg["content"]
            if len(content) > 200:
                content = content[:200] + "..."
            print(f"  [{msg['role']:10s}]: {content}")


## Step 6: LoRA Fine-tuning
使用 **LoRA + 4-bit 量化**对 Qwen2.5-7B-Instruct 进行 SFT 微调。
以下参数已针对 **Colab T4（16 GB 显存）** 调优，默认配置可有效防止 OOM。
若使用 A100，可适当调大 `MAX_LENGTH`、`r`、`batch_size` 以加快训练。

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

# ── 路径配置 ──────────────────────────────────────────────────────────────
# MODEL_NAME: HuggingFace 上的模型 ID，Colab 中直接从 HF 加载，无需本地下载
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
DATA_PATH  = "/content/data/food_recipes.jsonl"   # Step 4 生成的数据文件
OUTPUT_DIR = "/content/outputs/qwen25-7b-food-recipes-lora"  # adapter 保存路径

# ── 关键超参（防 OOM 首选值） ─────────────────────────────────────────────
# MAX_LENGTH: 单条序列最大 token 数。越大越费显存，T4 建议 ≤ 1024；A100 可上 2048
MAX_LENGTH = 1024

# ── 1. 4-bit 量化配置（BitsAndBytes NF4） ────────────────────────────────
# 4-bit 量化可将 7B 模型显存从 ~14 GB 压缩到 ~5 GB，是 T4 上跑大模型的关键
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,               # 启用 4-bit 量化加载
    bnb_4bit_quant_type="nf4",       # NF4 量化类型，效果优于 fp4
    bnb_4bit_compute_dtype=torch.bfloat16,  # 计算时用 bfloat16，精度与速度均衡
    bnb_4bit_use_double_quant=True,  # 双重量化，额外节省 ~0.4 bits/参数
)

# ── 2. 加载 Tokenizer ─────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Qwen 无 pad_token，用 eos 代替

# ── 3. 加载数据集并划分训练/验证集 ───────────────────────────────────────
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
# test_size=0.02：取 2% 作为验证集，用于训练中间评估
dataset = dataset.train_test_split(test_size=0.02, seed=42)
train_dataset = dataset["train"]
eval_dataset  = dataset["test"]
print(f"训练集：{len(train_dataset)} 条，验证集：{len(eval_dataset)} 条")

# 格式化函数：将 messages 按 Qwen ChatML 格式拼接成训练文本
def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

# ── 4. 加载模型（4-bit 量化） ────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,  # 应用 4-bit 量化
    device_map="auto",              # 自动分配到可用 GPU/CPU
    trust_remote_code=True,
)
model.config.use_cache = False       # 训练时关闭 KV cache，节省显存

# ── 5. LoRA 配置 ──────────────────────────────────────────────────────────
# r (rank)     : LoRA 秩。越大拟合能力越强但显存增加；r=8 是保守稳妥的起点
# lora_alpha   : 缩放系数，通常设为 2*r；alpha/r 比值决定 LoRA 更新幅度
# lora_dropout : Dropout 概率，轻度正则化，防止 LoRA 层过拟合
# target_modules: 对哪些线性层插入 LoRA；覆盖 attention + FFN 效果较好
peft_config = LoraConfig(
    r=8,                             # 秩=8，T4 显存友好
    lora_alpha=16,                   # 缩放系数=16（= 2 × r）
    lora_dropout=0.05,               # 轻度 Dropout
    bias="none",                    # 不训练 bias 参数
    task_type="CAUSAL_LM",          # 因果语言模型任务
    target_modules=[                 # Qwen2 架构的全部线性层
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

# ── 6. 训练参数 ───────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # 批次大小：per_device=1 防止 OOM
    # gradient_accumulation_steps=16 等效于逻辑 batch_size=16，不额外占显存
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    num_train_epochs=1,              # 先跑 1 epoch，确认收敛后再增加
    learning_rate=2e-4,              # LoRA 常用学习率，比全参数微调约大 10 倍
    warmup_ratio=0.03,               # 前 3% 步做学习率预热，防止初始震荡
    lr_scheduler_type="cosine",     # 余弦退火，训练后期学习率平滑下降
    weight_decay=0.01,               # L2 正则化，轻度防止过拟合

    logging_steps=10,                # 每 10 步打印一次 loss
    evaluation_strategy="steps",    # 按步数做验证（与 transformers 各版本兼容）
    eval_steps=100,                  # 每 100 步验证一次
    save_steps=100,                  # 每 100 步保存一次 checkpoint
    save_total_limit=2,              # 最多保留 2 个 checkpoint，节省磁盘空间

    bf16=torch.cuda.is_available(),  # A100/H100 支持 bf16；T4 不支持，自动回退
    fp16=False,                      # bf16 与 fp16 互斥，只开一个
    gradient_checkpointing=True,     # 梯度检查点：以时间换显存（强烈建议开启）

    report_to="none",               # 不上报到 wandb/tensorboard，避免额外配置
    optim="adamw_torch",            # PyTorch 原生 AdamW 优化器
)

# ── 7. 初始化 SFTTrainer 并开始训练 ─────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,         # 传入 LoRA 配置
    processing_class=tokenizer,      # 新版 trl 用 processing_class 替代 tokenizer
    formatting_func=formatting_func, # 将 messages 格式化为训练文本
    max_seq_length=MAX_LENGTH,       # trl >= 0.7 使用 max_seq_length；
                                     # 若报错 unexpected keyword argument，说明 trl 版本过旧，
                                     # 改为 max_length=MAX_LENGTH 即可
)

# 开始训练（T4 约需 1~3 小时，取决于数据量）
print("🚀 开始训练...")
trainer.train()

# ── 8. 保存 LoRA Adapter ─────────────────────────────────────────────────
# 只保存 LoRA adapter（几十 MB），不保存完整模型（~14 GB）
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ 微调完成！LoRA adapter 已保存到 {OUTPUT_DIR}")


## Step 7: Check Training Artifacts
验证训练产物是否正常生成，检查 adapter 权重文件与 tokenizer 配置。

In [ ]:
import os

OUTPUT_DIR = "/content/outputs/qwen25-7b-food-recipes-lora"
print(f"=== 训练产物检查：{OUTPUT_DIR} ===")

if os.path.exists(OUTPUT_DIR):
    files = sorted(os.listdir(OUTPUT_DIR))
    for fname in files:
        fpath = os.path.join(OUTPUT_DIR, fname)
        size  = os.path.getsize(fpath) / (1024 * 1024)  # 换算为 MB
        print(f"  {fname:45s}  {size:.2f} MB")
    # 关键文件检查（缺失则训练可能未正常结束）
    must_have = ["adapter_model.safetensors", "adapter_config.json", "tokenizer_config.json"]
    print()
    for f in must_have:
        status = "✅" if f in files else "❌ 缺失！请检查训练是否正常完成"
        print(f"  {status}  {f}")
else:
    print(f"❌ 输出目录不存在：{OUTPUT_DIR}，请先执行 Step 6 训练 cell。")
